<b> Transform Orders Data - Explode Arrays
1. Access elements from the JSON object 
2. Deduplicate Array Elements 
3. Explode Arrays
4. Write the Transformed Data to Silver Schema

In [0]:
df_orders = spark.table("gizmobox.silver.py_orders")
display(df_orders)

<b> 1. Access elements from the JSON object
<br>
<column_name.object>

In [0]:
from pyspark.sql import functions as f

In [0]:
df_orders_normalized = (
    df_orders
    .select (
       "json_value.order_id",
       "json_value.order_status",
       "json_value.payment_method",
       "json_value.total_amount",
       "json_value.transaction_timestamp",
       "json_value.customer_id",
       "json_value.items"
    )
)
display(df_orders_normalized)


<b> 2. Deduplicate Array Elements

[Function Array Distinct](https://learn.microsoft.com/en-us/azure/databricks/pyspark/reference/functions/array_distinct)

In [0]:
df_orders_normalized = (
    df_orders
    .select(
        "json_value.order_id",
        "json_value.order_status",
        "json_value.payment_method",
        "json_value.total_amount",
        "json_value.transaction_timestamp",
        "json_value.customer_id",
        f.array_distinct("json_value.items").alias("items")
    )
)
display(df_orders_normalized)

<b> 3. Explode Arrays
<br>
[Function Explode](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/explode) 

In [0]:
df_orders_exploded = (
    df_orders_normalized
    .select (
        "order_id",
        "order_status",
        "payment_method",
        "total_amount",
        "transaction_timestamp",
        "customer_id",
        f.explode("items").alias("item")
    )
)

display(df_orders_exploded)

<b> 4. Write the Transformed Data to Silver Schema


In [0]:
df_order_items = (
    df_orders_exploded
    .select (
        "order_id",
        "order_status",
        "payment_method",
        "total_amount",
        "transaction_timestamp",
        "customer_id",
        "item.item_id",
        "item.item_name",
        "item.item_price",
        "item.item_quantity",
        "item.item_category",
        "item.item_detail"
    )
)

In [0]:
%sql
SELECT order_id,
       order_status,
       payment_method,
       total_amount,
       transaction_timestamp,
       customer_id,
       items.item_id,
       items.name,
       items.price,
       items.quantity,
       items.category,
       items.details.brand,
       items.details.color
    FROM tv_orders_exploded;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gizmobox.silver.orders
AS
SELECT order_id,
       order_status,
       payment_method,
       total_amount,
       transaction_timestamp,
       customer_id,
       items.item_id,
       items.name,
       items.price,
       items.quantity,
       items.category,
       items.details.brand,
       items.details.color
    FROM tv_orders_exploded;

In [0]:
%sql
SELECT * FROM gizmobox.silver.orders